In [1]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix, identity
from scipy.sparse.linalg import inv
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from multiprocessing import Pool, cpu_count
from tqdm import tqdm
from collections import defaultdict
import time
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix, identity
from scipy.sparse.linalg import inv
from sklearn.preprocessing import LabelEncoder
import time
from tqdm.auto import tqdm

In [12]:
def sample_dense_core(df, target_ratings=100_000, core_threshold=5, random_state=42):
    """
    Performs Density-Biased Core Sampling.

    Args:
        df (pd.DataFrame): The full ratings dataframe with columns ['user_id', 'item_id', 'rating'].
        target_ratings (int): The desired approximate number of ratings in the final sample.
        core_threshold (int): The minimum number of interactions for users/items to be kept in the core.
        random_state (int): Random state for reproducibility.

    Returns:
        pd.DataFrame: A dense, interconnected sample of the original dataframe.
    """
    print("--- Starting Dense Core Sampling ---")
    
    # 1. Iterative Filtering to find the dense core
    print(f"Finding the dense core with threshold >={core_threshold}...")
    n_users_before, n_items_before = df['user_id'].nunique(), df['item_id'].nunique()
    
    while True:
        before_len = len(df)
        
        # Filter items
        item_counts = df['item_id'].value_counts()
        items_to_keep = item_counts[item_counts >= core_threshold].index
        df = df[df['item_id'].isin(items_to_keep)]
        
        # Filter users
        user_counts = df['user_id'].value_counts()
        users_to_keep = user_counts[user_counts >= core_threshold].index
        df = df[df['user_id'].isin(users_to_keep)]

        if len(df) == before_len:
            break
            
    n_users_after, n_items_after = df['user_id'].nunique(), df['item_id'].nunique()
    print(f"Dense core found: {len(df)} ratings, {n_users_after} users, {n_items_after} items.")
    print(f"(Removed {n_users_before - n_users_after} users and {n_items_before - n_items_after} items)")

    # 2. User-Centric Activity Sampling from the core
    print(f"\nSampling users from the core to reach ~{target_ratings} ratings...")
    
    # Get the final user counts from the dense core
    final_user_counts = df['user_id'].value_counts()
    
    # Sort users by activity to prioritize those with more data
    active_users_sorted = final_user_counts.index.tolist()
    
    # Shuffle for randomness, but we could also just take the top N most active
    # A little shuffle prevents always picking the same superusers if the data doesn't change
    np.random.seed(random_state)
    np.random.shuffle(active_users_sorted)
    
    selected_users = []
    accumulated_ratings = 0
    for user_id in active_users_sorted:
        if accumulated_ratings >= target_ratings:
            break
        
        selected_users.append(user_id)
        accumulated_ratings += final_user_counts[user_id]

    print(f"Selected {len(selected_users)} users, accumulating {accumulated_ratings} ratings.")
    
    # 3. Final DataFrame Creation
    # This is the crucial step: get ALL ratings for the selected users
    final_df = df[df['user_id'].isin(selected_users)].reset_index(drop=True)
    
    print("--- Dense Core Sampling Complete ---")
    return final_df

In [13]:
# --- Your New Pipeline ---
# Load the full dataset
user_item_ratings_subratings = pd.read_csv('../data/filtered/hotelrec_filtered_00001.csv')
full_ratings_df = user_item_ratings_subratings[['user_id', 'item_id', 'rating']]
full_ratings_df = full_ratings_df.drop_duplicates(subset=['user_id', 'item_id'])

print("Full dataset info:")
print(f"Shape: {full_ratings_df.shape}")
print(f"Users: {full_ratings_df['user_id'].nunique()}")
print(f"Items: {full_ratings_df['item_id'].nunique()}")
print("-" * 20)

# Apply the smart sampling
ratings_df = sample_dense_core(full_ratings_df, target_ratings=100_000, core_threshold=5)

print("\nFinal sampled dataset info:")
print(f"Shape: {ratings_df.shape}")
print(f"Users: {ratings_df['user_id'].nunique()}")
print(f"Items: {ratings_df['item_id'].nunique()}")
ratings_per_user = ratings_df.shape[0] / ratings_df['user_id'].nunique()
print(f"Density (Avg Ratings/User): {ratings_per_user:.2f}")
print("-" * 20)

ratings_df['rating'] = ratings_df['rating'].astype(np.float16)
ratings_df['user_id'] = ratings_df['user_id'].astype(np.int32)
ratings_df['item_id'] = ratings_df['item_id'].astype(np.int32)

Full dataset info:
Shape: (27145744, 3)
Users: 4081761
Items: 364275
--------------------
--- Starting Dense Core Sampling ---
Finding the dense core with threshold >=5...
Dense core found: 19979950 ratings, 1928038 users, 310334 items.
(Removed 2153723 users and 53941 items)

Sampling users from the core to reach ~100000 ratings...
Selected 9715 users, accumulating 100006 ratings.
--- Dense Core Sampling Complete ---

Final sampled dataset info:
Shape: (100006, 3)
Users: 9715
Items: 61903
Density (Avg Ratings/User): 10.29
--------------------


In [2]:
# user_item_ratings_subratings = pd.read_csv('../data/filtered/hotelrec_filtered_00001.csv')
# ratings_df = user_item_ratings_subratings[['user_id', 'item_id', 'rating']]
# ratings_df = ratings_df.sample(n=100_000, random_state=42)
# # drop duplicates
# print(f"Original shape: {ratings_df.shape}")
# ratings_df = ratings_df.drop_duplicates(subset=['user_id', 'item_id'])
# print(f"Shape after dropping duplicates: {ratings_df.shape}")

# # load data as half-precision floats and integers as half ints to save memory
# ratings_df['rating'] = ratings_df['rating'].astype(np.float16)
# ratings_df['user_id'] = ratings_df['user_id'].astype(np.int32)
# ratings_df['item_id'] = ratings_df['item_id'].astype(np.int32)
# ratings_df.info()

Original shape: (100000, 3)
Shape after dropping duplicates: (99003, 3)
<class 'pandas.core.frame.DataFrame'>
Index: 99003 entries, 4647116 to 19020529
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   user_id  99003 non-null  int32  
 1   item_id  99003 non-null  int32  
 2   rating   99003 non-null  float16
dtypes: float16(1), int32(2)
memory usage: 1.7 MB


In [3]:
# --- Provided / OLD EASE Class ---
class EASE:
    def __init__(self):
        self.user_enc = LabelEncoder()
        self.item_enc = LabelEncoder()

    def _get_users_and_items(self, df):
        users = self.user_enc.fit_transform(df.loc[:, 'user_id'])
        items = self.item_enc.fit_transform(df.loc[:, 'item_id'])
        return users, items

    def fit(self, df, lambda_: float = 0.5, implicit=True):
        """
        df: pandas.DataFrame with columns user_id, item_id and (rating)
        lambda_: l2-regularization term
        implicit: if True, ratings are ignored and taken as 1, else normalized ratings are used
        """
        users, items = self._get_users_and_items(df)
        values = (
            np.ones(df.shape[0])
            if implicit
            else df['rating'].to_numpy() / df['rating'].max()
        )

        X = csr_matrix((values, (users, items)))
        self.X = X

        G = X.T.dot(X).toarray()
        diagIndices = np.diag_indices(G.shape[0])
        G[diagIndices] += lambda_
        P = np.linalg.inv(G)
        B = P / (-np.diag(P))
        B[diagIndices] = 0

        self.B = B
        self.pred = X.dot(B)

    def predict(self, train, users, items, k):
        items = self.item_enc.transform(items)
        dd = train.loc[train.user_id.isin(users)]
        dd['ci'] = self.item_enc.transform(dd.item_id)
        dd['cu'] = self.user_enc.transform(dd.user_id)
        g = dd.groupby('cu')
        with Pool(cpu_count()) as p:
            user_preds = p.starmap(
                self.predict_for_user,
                [(user, group, self.pred[user, :], items, k) for user, group in g],
            )
        df = pd.concat(user_preds)
        df['item_id'] = self.item_enc.inverse_transform(df['item_id'])
        df['user_id'] = self.user_enc.inverse_transform(df['user_id'])
        return df

    @staticmethod
    def predict_for_user(user, group, pred, items, k):
        watched = set(group['ci'])
        candidates = [item for item in items if item not in watched]
        pred = np.take(pred, candidates)
        res = np.argpartition(pred, -k)[-k:]
        r = pd.DataFrame(
            {
                "user_id": [user] * len(res),
                "item_id": np.take(candidates, res),
                "score": np.take(pred, res),
            }
        ).sort_values('score', ascending=False)
        return r

In [14]:
# --- Optimmized EASE ---- 
class EASE:
    def __init__(self):
        self.user_enc = LabelEncoder()
        self.item_enc = LabelEncoder()
        self.B = None # This will be our item-item similarity matrix
        self.X = None # This will be our user-item training matrix

    def fit(self, df, lambda_: float = 100, implicit=True):
        """
        Trains the EASE model.
        
        Key Optimizations:
        1. All matrix operations are kept in sparse format to conserve memory.
        2. Uses scipy's sparse inverse, which is much faster and more memory-efficient.
        3. Avoids pre-calculating all predictions, saving massive amounts of memory.
        """
        print("Fitting EASE model...")
        
        # --- Data Encoding ---
        df['encoded_user'] = self.user_enc.fit_transform(df['user_id'])
        df['encoded_item'] = self.item_enc.fit_transform(df['item_id'])
        
        # --- User-Item Matrix Creation ---
        values = np.ones(df.shape[0]) if implicit else df['rating'].to_numpy() / df['rating'].max()
        self.X = csr_matrix((values, (df['encoded_user'], df['encoded_item'])))
        
        # --- Gram Matrix and Inversion (Memory & Speed Optimized) ---
        # The Gram matrix G is computed and kept as a sparse matrix (csc_matrix is efficient for this)
        G = self.X.T.dot(self.X).tocsc()
        
        # Add the regularization term to the diagonal
        G += identity(G.shape[0]) * lambda_
        
        # Invert the sparse matrix. This is the most compute-intensive part.
        P = inv(G)
        
        # Convert to a dense array ONLY for the final division and diagonal manipulation
        P_dense = P.toarray()
        
        # Calculate the final item-item similarity matrix B
        # The diagonal of P contains the negative inverse of the item "self-similarity"
        B = P_dense / (-np.diag(P_dense))
        np.fill_diagonal(B, 0)
        
        self.B = B.astype(np.float16) # Store as float32 to save memory
        print("Fit complete.")

    def predict(self, train_df, users, k, batch_size=2048):
        """
        Generates top-k recommendations for a list of users.
        
        Key Optimizations:
        1. NO MULTIPROCESSING: Uses a fast, vectorized batch processing approach.
        2. Batched Prediction: Calculates scores in small batches to keep memory usage low and constant.
        3. Efficient Watched Item Exclusion: Uses a pre-computed dictionary for fast lookups.
        4. Vectorized Top-K: Uses NumPy's argpartition for fast top-k selection across the entire batch.
        """
        if self.B is None:
            raise RuntimeError("The model has not been fitted yet. Call fit() first.")
            
        print(f"Generating top-{k} recommendations for {len(users)} users...")
        
        # --- Pre-computation for Watched Item Exclusion ---
        # This is much faster than doing groupby.apply(list)
        watched_items_dict = train_df.groupby('user_id')['encoded_item'].apply(list).to_dict()
        
        # Get the internal integer encodings for the users we want to predict for
        encoded_users_predict = self.user_enc.transform(users)
        
        all_recs = []
        # --- Batch Processing Loop ---
        for i in tqdm(range(0, len(encoded_users_predict), batch_size), desc="Predicting in Batches"):
            batch_user_indices = encoded_users_predict[i:i+batch_size]
            
            # --- Score Calculation (for the batch) ---
            # This is the core prediction step: (Users x Items) @ (Items x Items)
            batch_scores = self.X[batch_user_indices, :].dot(self.B)

            # --- Watched Item Exclusion (for the batch) ---
            for j, user_idx in enumerate(batch_user_indices):
                user_id = self.user_enc.inverse_transform([user_idx])[0]
                items_to_exclude = watched_items_dict.get(user_id, [])
                if items_to_exclude:
                    batch_scores[j, items_to_exclude] = -np.inf

            # --- Top-K Selection (for the batch) ---
            # np.argpartition is much faster than np.argsort for finding top-k
            top_k_indices = np.argpartition(batch_scores, -k, axis=1)[:, -k:]
            
            # We need both the scores and the original user indices for the results
            top_k_scores = np.take_along_axis(batch_scores, top_k_indices, axis=1)
            batch_user_ids = np.repeat(users[i:i+len(batch_user_indices)], k)
            
            # --- Formatting results for the batch ---
            batch_df = pd.DataFrame({
                'user_id': batch_user_ids,
                'item_id': self.item_enc.inverse_transform(top_k_indices.flatten()),
                'score': top_k_scores.flatten()
            })
            all_recs.append(batch_df)
            
        # --- Final Combination ---
        final_recs = pd.concat(all_recs)
        final_recs = final_recs.sort_values(['user_id', 'score'], ascending=[True, False])
        print("Prediction complete.")
        return final_recs

In [15]:
class FastEvaluator:
    def __init__(self, test_df, k_values=[1, 3, 5, 10]):
        """
        Initializes the single-threaded, high-performance evaluator.
        """
        print("Initializing FastEvaluator...")
        self.k_values = sorted(k_values)
        self.max_k = max(self.k_values)

        # Create the ground truth dictionary: {user_id: {item_id_1, ...}} for O(1) lookups
        self.ground_truth_sets = test_df.groupby('user_id')['item_id'].apply(set).to_dict()
        
        # Pre-calculate Ideal Discounted Cumulative Gain (IDCG) table to avoid log calculations in the loop
        self.idcg_table = {i: sum(1 / np.log2(rank + 2) for rank in range(i)) for i in range(1, self.max_k + 1)}
        print("Evaluator ready.")

    def evaluate(self, recommendations_df):
        """
        Calculates Precision, Recall, and nDCG for all specified k values in a single pass.
        """
        print(f"Evaluating recommendations for k={self.k_values}...")
        
        # Group recommendations by user once for efficient lookup
        recs_by_user = recommendations_df.groupby('user_id')['item_id'].apply(list).to_dict()
        
        # Get list of users that are in both test set and recommendations
        test_users = [user for user in self.ground_truth_sets if user in recs_by_user]
        
        # Dictionaries to hold the sum of scores over all users
        precision_sum = {k: 0 for k in self.k_values}
        recall_sum = {k: 0 for k in self.k_values}
        ndcg_sum = {k: 0 for k in self.k_values}
        num_users_evaluated = 0

        # --- Main Evaluation Loop (Single-Threaded) ---
        for user_id in tqdm(test_users, desc="Evaluating Users"):
            true_items = self.ground_truth_sets.get(user_id)
            recommended_items = recs_by_user.get(user_id)

            if not true_items or not recommended_items:
                continue
            
            num_users_evaluated += 1
            num_true_items = len(true_items)
            
            hits = 0
            dcg = 0
            
            # --- Single-Pass Calculation for all K ---
            # We iterate through the top max_k recommendations just ONCE per user
            for i, item_id in enumerate(recommended_items[:self.max_k]):
                rank = i + 1
                if item_id in true_items:
                    hits += 1
                    dcg += 1 / np.log2(rank + 1)
                
                # At each k we care about, record the current metric values
                if rank in self.k_values:
                    precision_sum[rank] += hits / rank
                    recall_sum[rank] += hits / num_true_items
                    idcg = self.idcg_table.get(min(num_true_items, rank), 0)
                    ndcg_sum[rank] += dcg / idcg if idcg > 0 else 0
        
        # Calculate final average metrics
        avg_precision = {k: precision_sum[k] / num_users_evaluated if num_users_evaluated > 0 else 0 for k in self.k_values}
        avg_recall = {k: recall_sum[k] / num_users_evaluated if num_users_evaluated > 0 else 0 for k in self.k_values}
        avg_ndcg = {k: ndcg_sum[k] / num_users_evaluated if num_users_evaluated > 0 else 0 for k in self.k_values}
        
        # --- Format and display results in a clean table ---
        results_df = pd.DataFrame({
            'k': self.k_values,
            'Precision@k': [avg_precision[k] for k in self.k_values],
            'Recall@k': [avg_recall[k] for k in self.k_values],
            'nDCG@k': [avg_ndcg[k] for k in self.k_values]
        }).set_index('k')
        
        print("\n--- Evaluation Results ---")
        print(results_df)
        
        return results_df

In [16]:
# --- Train-Test Split ---
train_df, test_df = train_test_split(ratings_df, test_size=0.05, random_state=42)

# Ensure all users and items in the test set are also in the training set
test_df = test_df[test_df['user_id'].isin(train_df['user_id'])]
test_df = test_df[test_df['item_id'].isin(train_df['item_id'])]

In [17]:
# --- Instantiate and Train the Model ---
ease_model = EASE()
ease_model.fit(train_df, lambda_=250, implicit=False)

Fitting EASE model...


In [8]:
# save the model
import pickle
with open('ease_model_l250_100k.pkl', 'wb') as f:
    pickle.dump(ease_model, f)

In [11]:
# --- 4. Generating Recommendations ---
print("\n--- Predicting ---")
test_users = test_df['user_id'].unique()
# We must recommend at least max(k) items to evaluate properly
k_to_recommend = 10
start_time = time.time()
recommendations = ease_model.predict(train_df, test_users, k=k_to_recommend)
print(f"Total prediction time: {time.time() - start_time:.2f} seconds")

# --- 5. Fast, Single-Threaded Evaluation ---
print("\n--- Evaluating ---")
k_values_to_evaluate = [1, 3, 5, 10]
start_time = time.time()
evaluator = FastEvaluator(test_df, k_values=k_values_to_evaluate)
results_summary = evaluator.evaluate(recommendations)
print(f"Total evaluation time: {time.time() - start_time:.2f} seconds")


--- Predicting ---
Generating top-10 recommendations for 83 users...


Predicting in Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Prediction complete.
Total prediction time: 14.74 seconds

--- Evaluating ---
Initializing FastEvaluator...
Evaluator ready.
Evaluating recommendations for k=[1, 3, 5, 10]...


Evaluating Users:   0%|          | 0/83 [00:00<?, ?it/s]


--- Evaluation Results ---
    Precision@k  Recall@k  nDCG@k
k                                
1           0.0       0.0     0.0
3           0.0       0.0     0.0
5           0.0       0.0     0.0
10          0.0       0.0     0.0
Total evaluation time: 0.01 seconds


In [ ]:
# --- Evaluation Metrics ---
def precision_at_k(recommendations, ground_truth, k):
    """Calculates precision@k."""
    merged = pd.merge(recommendations, ground_truth, on=['user_id', 'item_id'], how='left', indicator=True)
    hits = merged[merged['_merge'] == 'both'].groupby('user_id').size().reset_index(name='hits')
    precision = (hits['hits'] / k).mean()
    return precision

def recall_at_k(recommendations, ground_truth, k):
    """Calculates recall@k."""
    ground_truth_counts = ground_truth.groupby('user_id').size().reset_index(name='total_relevant')
    merged = pd.merge(recommendations, ground_truth, on=['user_id', 'item_id'], how='inner')
    hits = merged.groupby('user_id').size().reset_index(name='hits')
    
    merged_counts = pd.merge(hits, ground_truth_counts, on='user_id')
    recall = (merged_counts['hits'] / merged_counts['total_relevant']).mean()
    return recall

def map_at_k(recommendations, ground_truth, k):
    """Calculates Mean Average Precision@k."""
    ap_sum = 0
    # Group recommendations and ground truth by user
    rec_grouped = recommendations.groupby('user_id')
    gt_grouped = ground_truth.groupby('user_id')
    
    for user_id, group in rec_grouped:
        if user_id in gt_grouped.groups:
            relevant_items = set(gt_grouped.get_group(user_id)['item_id'])
            recommended_items = group['item_id'].tolist()
            
            hits = 0
            precision_sum = 0
            for i, item_id in enumerate(recommended_items):
                if item_id in relevant_items:
                    hits += 1
                    precision_sum += hits / (i + 1)
            
            ap_sum += precision_sum / min(len(relevant_items), k)
            
    return ap_sum / len(rec_grouped)

def ndcg_at_k(recommendations, ground_truth, k):
    """Calculates Normalized Discounted Cumulative Gain@k."""
    ndcg_sum = 0
    # Group recommendations and ground truth by user
    rec_grouped = recommendations.groupby('user_id')
    gt_grouped = ground_truth.groupby('user_id')
    
    for user_id, group in rec_grouped:
        if user_id in gt_grouped.groups:
            relevant_items = set(gt_grouped.get_group(user_id)['item_id'])
            recommended_items = group['item_id'].tolist()
            
            # Calculate DCG
            dcg = 0
            for i, item_id in enumerate(recommended_items):
                if item_id in relevant_items:
                    dcg += 1 / np.log2(i + 2) # +2 because ranks are 1-based
            
            # Calculate IDCG
            idcg = sum([1 / np.log2(i + 2) for i in range(min(len(relevant_items), k))])
            
            ndcg_sum += dcg / idcg if idcg > 0 else 0
            
    return ndcg_sum / len(rec_grouped)


# --- Run Evaluation ---
ground_truth = test_df[['user_id', 'item_id']]

precision = precision_at_k(recommendations, ground_truth, k)
recall = recall_at_k(recommendations, ground_truth, k)
map_k = map_at_k(recommendations, ground_truth, k)
ndcg = ndcg_at_k(recommendations, ground_truth, k)

# --- Display Results ---
print("--- EASE Model Evaluation ---")
print(f"Top-{k} Recommendations")
print(f"Precision@{k}: {precision:.4f}")
print(f"Recall@{k}:    {recall:.4f}")
print(f"MAP@{k}:       {map_k:.4f}")
print(f"NDCG@{k}:      {ndcg:.4f}")